# 02 — Train Qwen3-8B: Translation (Nepali ↔ English)

- **Model:** `unsloth/Qwen3-8B-unsloth-bnb-4bit`
- **Dataset:** opus-100 ne-en (5,000 pairs)
- **Method:** QLoRA r=16 via Unsloth
- **Output:** `iwasbinod/qwen3-8b-nepali-translation-qlora`

In [ ]:
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git -q
!pip install "trl>=0.12.0,<0.16.0" peft accelerate bitsandbytes sacrebleu rouge-score -q

In [ ]:
import sys
sys.path.append('/kaggle/working/final_year_proj_finetuning')

from src.data_utils import load_jsonl
from src.training_utils import load_model, apply_qlora, build_trainer, save_and_push
from src.evaluation import evaluate_translation
from src.utils import set_seed, hf_login, print_gpu_memory

set_seed(42)
print_gpu_memory()

# Login to HuggingFace (add your token to Kaggle secrets)
import os
hf_login(os.environ.get('HF_TOKEN'))

In [ ]:
# Load training data
train_data = load_jsonl('data/processed/translation_train.jsonl')
val_data   = load_jsonl('data/processed/translation_val.jsonl')
print(f'Train: {len(train_data)}, Val: {len(val_data)}')

# Preview
print('\nSample prompt:')
print(train_data[0]['text'])

In [ ]:
# Load model + apply QLoRA
model, tokenizer = load_model('qwen3-8b')
model = apply_qlora(model, r=16, lora_alpha=32)
print_gpu_memory()

In [ ]:
# Build trainer and train
trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    train_data=train_data,
    output_dir='outputs/adapters/qwen3-8b-nepali-translation-qlora',
    num_epochs=3,
    batch_size=2,
    grad_accum=4,
    lr=2e-4,
    max_seq_length=512,
)

print('Starting training...')
trainer_stats = trainer.train()
print('Training complete!')
print(f'Training loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# Evaluate fine-tuned model
print('Evaluating fine-tuned model...')
results = evaluate_translation(model, tokenizer, val_data, n_samples=100)
print(f'\nFine-tuned BLEU: {results["bleu"]}')
print('\nSample predictions:')
for i in range(3):
    print(f'  Ref:  {results["references"][i]}')
    print(f'  Pred: {results["hypotheses"][i]}')
    print()

In [ ]:
# Save adapter and push to HuggingFace
repo_id = save_and_push(model, tokenizer, model_key='qwen3-8b', task='translation')
print(f'Adapter available at: https://huggingface.co/{repo_id}')